In [2]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# load data

In [3]:
df = pd.read_csv('/Users/connorhall/datasets/inst414/module 3 assignment/Public.csv',
                 usecols=['SPECIES', 'AIRPORT', 'STATE', 'FAAREGION'])
df = df.dropna()

# remove unknown species
df = df[(~df['SPECIES'].str.contains('unknown|perching birds', case=False)) &
        (~df['AIRPORT'].str.contains('unknown', case=False))]

df

,AIRPORT,STATE,FAAREGION,SPECIES
8,NORMAN Y. MINETA SAN JOSE INTL ARPT,CA,AWP,American robin
11,ALEXANDRIA INTL,LA,ASW,Blackbirds
13,SYRACUSE HANCOCK INTL,NY,AEA,Gulls
14,OAKLAND COUNTY INTL,MI,AGL,Mourning dove
15,LOS ANGELES INTL,CA,AWP,American kestrel
...,...,...,...,...
339266,EVANSVILLE REGIONAL,IN,AGL,Chimney swift
339267,SEATTLE-TACOMA INTL,WA,ANM,Cedar waxwing
339268,CHICAGO O'HARE INTL ARPT,IL,AGL,Yellow-rumped warbler
339269,ROANOKE REGNL ARPT/WOODRUM FIELD,VA,AEA,European starling


# filter

In [4]:
# remove outlier states- vastly different species
outlier_loc = ['AB', 'ON', 'PR', 'FN', 'QC', 'BC', 'AS', 'VI', 'AK',
                'UM', 'GU', 'MH', 'SK', 'NL', 'NS', 'MB', 'MP', 'HI']
df = df[~df['STATE'].isin(outlier_loc)]

df = df[df.groupby('SPECIES')['SPECIES'].transform('count') >= 100]
df = df[df.groupby('AIRPORT')['AIRPORT'].transform('count') >= 100]

df = df[['SPECIES', 'AIRPORT']]

# calculate collision counts

In [7]:
counts = pd.crosstab(df['SPECIES'], df['AIRPORT']).stack('AIRPORT')
counts

SPECIES                AIRPORT                     
American barn owl      ABERDEEN REGIONAL ARPT          0
                       ABRAHAM LINCOLN CAPITAL ARPT    0
                       ADDISON AIRPORT                 0
                       ALBANY INTL                     0
                       ALBUQUERQUE INTL SUNPORT        1
                                                      ..
Yellow-rumped warbler  WILLOW RUN ARPT                 0
                       WILMINGTON INTL                 3
                       WORCESTER REGIONAL              0
                       YAMPA VALLEY                    1
                       YEAGER ARPT                     0
Length: 36708, dtype: int64

### separate airport and species index

In [8]:
species = counts.index.get_level_values('SPECIES').tolist()
airports = counts.index.get_level_values('AIRPORT').tolist()

counts_df = pd.DataFrame({'Species': species, 'Airport': airports})
counts_df['Count'] = counts.values
counts_df

,Species,Airport,Count
0,American barn owl,ABERDEEN REGIONAL ARPT,0
1,American barn owl,ABRAHAM LINCOLN CAPITAL ARPT,0
2,American barn owl,ADDISON AIRPORT,0
3,American barn owl,ALBANY INTL,0
4,American barn owl,ALBUQUERQUE INTL SUNPORT,1
...,...,...,...
36703,Yellow-rumped warbler,WILLOW RUN ARPT,0
36704,Yellow-rumped warbler,WILMINGTON INTL,3
36705,Yellow-rumped warbler,WORCESTER REGIONAL,0
36706,Yellow-rumped warbler,YAMPA VALLEY,1


### add collision counts to new dataframe format

In [9]:
# create lists of unique species names and airport names
unique_sp = sorted(list(set(species)))
unique_air = sorted(list(set(airports)))

# create list of collision count lists
counts_list = []
for sp in unique_sp:
    sp_filter = counts_df[counts_df['Species'] == sp]
    counts_list.append(sp_filter['Count'].tolist())
    
# create new dataframe format
collisions_df = pd.DataFrame(counts_list, columns=unique_air, index=unique_sp)
collisions_df

,ABERDEEN REGIONAL ARPT,ABRAHAM LINCOLN CAPITAL ARPT,ADDISON AIRPORT,ALBANY INTL,ALBUQUERQUE INTL SUNPORT,ASHEVILLE REGIONAL ARPT,ASPEN-PITKIN COUNTY ARPT/SARDY FIELD,ATLANTIC CITY INTL,AUGUSTA REGIONAL ARPT AT BUSH FLD,AUSTIN STRAUBEL INTL,...,WESTCHESTER COUNTY ARPT,WICHITA DWIGHT D EISENHOWER NATL ARPT,WILKES-BARRE/SCRANTON INTL,WILL ROGERS WORLD ARPT,WILLIAM P HOBBY ARPT,WILLOW RUN ARPT,WILMINGTON INTL,WORCESTER REGIONAL,YAMPA VALLEY,YEAGER ARPT
American barn owl,0,0,0,0,1,1,0,0,1,0,...,0,0,0,3,14,0,0,0,0,0
American coot,1,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
American crow,0,1,2,8,0,4,5,2,0,1,...,7,1,2,0,2,0,0,1,0,20
American golden-plover,0,0,0,3,0,0,0,0,0,1,...,1,0,0,0,2,1,0,0,0,0
American goldfinch,0,1,0,0,0,0,0,1,0,1,...,3,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Woodchuck,0,0,0,6,0,1,0,0,0,0,...,2,0,1,0,0,1,0,0,0,0
Yellow warbler,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
Yellow-bellied sapsucker,0,0,0,0,0,0,0,1,0,0,...,6,0,1,0,0,0,1,1,0,0
Yellow-crowned night heron,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,10,0,0,0,0,0


In [ ]:
vals = []
for ls in collisions_df.values.tolist():
    vals += ls

# percentage of values that are 0
int(pd.DataFrame(vals).value_counts()[0]) / (len(collisions_df.index) * len(collisions_df.columns))

0.6597471940721369

### verify output

In [12]:
# compare sum of collisions from counts_df and collisions_df
counts_1 = counts_df.groupby('Species')['Count'].sum().values.tolist()
counts_2 = collisions_df.sum(axis=1).values.tolist()
counts_1 == counts_2

True

# clustering

In [13]:
hellinger_df = np.sqrt(collisions_df.div(collisions_df.sum(axis=1), axis=0))

pca = PCA(n_components=0.90)
pca_data = pca.fit_transform(hellinger_df)

cluster_model = KMeans(n_clusters=5)
labels = cluster_model.fit_predict(pca_data)

copy_df = collisions_df.copy()
copy_df['label'] = labels

In [14]:
pca_data

array([[ 0.24068549, -0.2046575 ,  0.18640518, ..., -0.02508591,
        -0.01924243, -0.01145054],
       [ 0.20043393, -0.2149916 ,  0.01861552, ..., -0.05102255,
        -0.0009497 ,  0.03769559],
       [-0.1154717 , -0.091789  , -0.11048721, ...,  0.00910863,
        -0.06705121,  0.03745606],
       ...,
       [-0.45236502, -0.15416101,  0.06029851, ...,  0.03774727,
        -0.03867367,  0.03447276],
       [-0.01452134,  0.37527715,  0.39717582, ...,  0.01409734,
        -0.0585784 , -0.04559588],
       [-0.27305812, -0.0790697 ,  0.12419021, ..., -0.02422143,
        -0.01523378, -0.04868054]], shape=(161, 58))

In [15]:
pd.DataFrame(labels).value_counts()

0
2    50
3    41
1    36
4    21
0    13
Name: count, dtype: int64